# NB02 — Price History and Major Shocks
**Global Oil Market Analysis**

Oil has been priced in US dollars since 1945. In that time it has gone from $1.63/bbl to nearly $140/bbl and back again — multiple times. Each swing had a cause. This notebook traces 60+ years of price history, identifies the moments that broke the trend, quantifies the severity of each shock, and closes with the EIA's forward projection through 2027.

---

## Sections
1. Full price history (1960-2026)
2. Major price shocks — annotated timeline
3. Price volatility by era
4. Regional price divergence (World vs Europe vs Asia)
5. Rolling correlation — does price follow supply or demand?
6. EIA forecast context through 2027


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)

DATA_DIR = './data/processed/'

prices      = pd.read_parquet(DATA_DIR + 'prices.parquet')
master      = pd.read_parquet(DATA_DIR + 'master_actuals.parquet')

print(f'Prices loaded: {prices.shape} | {prices.index.min().date()} to {prices.index.max().date()}')
print(f'Master loaded: {master.shape}')

Prices loaded: (794, 5) | 1960-01-31 to 2026-02-28
Master loaded: (398, 44)


## 1. Full Price History (1960-2026)

In [2]:
# Major historical events to annotate on the chart
events = [
    {'date': '1973-10', 'label': 'OPEC Embargo',         'color': 'red'},
    {'date': '1979-01', 'label': 'Iran Revolution',      'color': 'red'},
    {'date': '1980-09', 'label': 'Iraq-Iran War',        'color': 'red'},
    {'date': '1986-01', 'label': 'Saudi Price War',      'color': 'orange'},
    {'date': '1990-08', 'label': 'Gulf War',             'color': 'red'},
    {'date': '1998-12', 'label': 'Asian Crisis low',     'color': 'orange'},
    {'date': '2004-01', 'label': 'China demand surge',   'color': 'blue'},
    {'date': '2008-07', 'label': '$147 peak',            'color': 'green'},
    {'date': '2008-12', 'label': 'Financial Crisis',     'color': 'red'},
    {'date': '2014-06', 'label': 'Shale supply glut',   'color': 'orange'},
    {'date': '2016-01', 'label': 'OPEC+ formed',        'color': 'purple'},
    {'date': '2020-04', 'label': 'COVID -$37/bbl',      'color': 'red'},
    {'date': '2022-06', 'label': 'Ukraine war spike',   'color': 'red'},
]

fig = go.Figure()

# Main price line
fig.add_trace(go.Scatter(
    x=prices.index, y=prices['price_world'],
    mode='lines', name='World avg (USD/bbl)',
    line=dict(color='#2563EB', width=1.5),
    fill='tozeroy', fillcolor='rgba(37,99,235,0.08)'
))

# 12-month rolling average
rolling = prices['price_world'].rolling(12).mean()
fig.add_trace(go.Scatter(
    x=prices.index, y=rolling,
    mode='lines', name='12-month avg',
    line=dict(color='#F59E0B', width=2, dash='dot')
))

# Annotate major events
for ev in events:
    try:
        dt  = pd.Timestamp(ev['date'])
        val = prices.loc[prices.index >= dt, 'price_world'].iloc[0]
        fig.add_vline(x=dt, line_width=1, line_dash='dash',
                      line_color=ev['color'], opacity=0.5)
        fig.add_annotation(
            x=dt, y=val + 8,
            text=ev['label'], showarrow=False,
            textangle=-90, font=dict(size=9, color=ev['color']),
            xanchor='left'
        )
    except IndexError:
        pass

fig.update_layout(
    title='Oil Price History 1960-2026 — World Average (USD/bbl)',
    xaxis_title='Date', yaxis_title='Price (USD/bbl)',
    hovermode='x unified',
    template='plotly_white',
    height=500, width=1100,
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

## 2. Price Shocks — Quantifying the Severity

In [3]:
# Identify the 10 biggest single-month price drops and spikes
prices['mom_pct'] = prices['price_world'].pct_change() * 100

print('10 BIGGEST MONTHLY PRICE DROPS:')
drops = prices['mom_pct'].nsmallest(10)
print(drops.to_string())

print()
print('10 BIGGEST MONTHLY PRICE SPIKES:')
spikes = prices['mom_pct'].nlargest(10)
print(spikes.to_string())

10 BIGGEST MONTHLY PRICE DROPS:
date
2020-03-31   -39.63
1986-02-28   -35.52
2020-04-30   -34.65
2008-10-31   -27.06
2008-11-30   -25.75
2008-12-31   -23.41
2015-01-31   -22.40
2000-12-31   -22.04
2014-12-31   -21.16
1986-03-31   -20.52

10 BIGGEST MONTHLY PRICE SPIKES:
date
1974-01-31   217.07
1990-08-31    53.79
1973-10-31    51.85
1979-05-31    49.39
2020-05-31    44.37
1986-08-31    42.46
1971-01-31    35.54
2020-06-30    29.88
1990-09-30    23.86
1999-03-31    22.44


In [13]:
# Peak-to-trough analysis for each major crash
crashes = [
    {'name': '1985-1986 Saudi Price War',  'peak': '1985-11', 'trough': '1986-07'},
    {'name': '1990-1998 Post-Gulf decline','peak': '1990-10', 'trough': '1998-12'},
    {'name': '2008 Financial Crisis',      'peak': '2008-07', 'trough': '2008-12'},
    {'name': '2014-2016 Shale Crash',      'peak': '2014-06', 'trough': '2016-01'},
    {'name': '2020 COVID Crash',           'peak': '2020-01', 'trough': '2020-04'},
    {'name': '2022-2023 Energy unwind',    'peak': '2022-06', 'trough': '2023-06'},
]

crash_summary = []
for c in crashes:
    peak_price   = prices.loc[c['peak'],   'price_world'].iloc[0]
    trough_price = prices.loc[c['trough'], 'price_world'].iloc[0]
    pct_drop     = (trough_price - peak_price) / peak_price * 100
    crash_summary.append({
        'event':          c['name'],
        'peak_date':      c['peak'],
        'peak_price':     round(peak_price, 2),
        'trough_date':    c['trough'],
        'trough_price':   round(trough_price, 2),
        'pct_drop':       round(pct_drop, 1),
    })

pd.DataFrame(crash_summary)

,event,peak_date,peak_price,trough_date,trough_price,pct_drop
0,1985-1986 Saudi Price War,1985-11,28.60,1986-07,9.62,-66.40
1,1990-1998 Post-Gulf decline,1990-10,34.50,1998-12,10.41,-69.80
2,2008 Financial Crisis,2008-07,132.83,2008-12,41.34,-68.90
3,2014-2016 Shale Crash,2014-06,108.37,2016-01,29.78,-72.50
4,2020 COVID Crash,2020-01,61.63,2020-04,21.04,-65.90
5,2022-2023 Energy unwind,2022-06,116.80,2023-06,73.26,-37.30


## 3. Price Volatility by Era

In [23]:
# Volatility = standard deviation of monthly % change within each era
era_stats = prices.groupby('price_era').agg(
    avg_price   = ('price_world', 'mean'),
    min_price   = ('price_world', 'min'),
    max_price   = ('price_world', 'max'),
    volatility  = ('mom_pct',    'std'),
    months      = ('price_world', 'count'),
).round(2).reset_index()

era_stats['price_range'] = era_stats['max_price'] - era_stats['min_price']

# Sort chronologically — extract the leading number from the era label
era_stats['sort_key'] = era_stats['price_era'].str.extract(r'^(\d+)\.').astype(int)
era_stats = era_stats.sort_values('sort_key').drop(columns='sort_key').reset_index(drop=True)

print(era_stats.to_string(index=False))

                        price_era  avg_price  min_price  max_price  volatility  months  price_range
          1. Pre-OPEC Era (<1973)       1.47       1.21       1.87        3.05     156         0.66
    2. OPEC Shock Era (1973-1985)      21.75       2.08      40.75       18.70     156        38.67
3. Price Collapse Era (1986-1998)      17.61       9.62      34.50        9.50     156        24.88
 4. China Demand Boom (1999-2007)      39.00      10.75      91.34        8.15     108        80.59
  5. Financial Crisis (2008-2009)      79.37      41.34     132.83       12.84      24        91.49
    6. High Price Era (2010-2014)      97.67      60.70     117.78        5.75      60        57.08
   7. Shale Crash Era (2015-2019)      55.22      29.78      76.73        8.84      60        46.95
         8. COVID Era (2020-2021)      55.16      21.04      82.06       17.19      24        61.02
     9. Energy Crisis (2022-2023)      88.93      73.26     116.80        8.84      24        43.54


In [6]:
fig = px.bar(
    era_stats, x='price_era', y='volatility',
    color='avg_price',
    color_continuous_scale='RdYlGn_r',
    title='Price Volatility by Era (Std Dev of Monthly % Change)',
    labels={'price_era': 'Era', 'volatility': 'Volatility (std dev, %)', 'avg_price': 'Avg Price ($)'}
)
fig.update_layout(xaxis_tickangle=-30, template='plotly_white', height=450)
fig.show()

## 4. Regional Price Divergence

In [7]:
# How far does Asia diverge from the world average?
# North America data starts later (WTI vs Brent spread becomes visible)
regional = prices.dropna(subset=['price_europe', 'price_asia'])

fig = go.Figure()

for col, name, color in [
    ('price_world',         'World avg', '#1E40AF'),
    ('price_europe',        'Europe',    '#059669'),
    ('price_asia',          'Asia',      '#DC2626'),
    ('price_north_america', 'N. America','#D97706'),
]:
    subset = prices.dropna(subset=[col])
    fig.add_trace(go.Scatter(
        x=subset.index, y=subset[col],
        mode='lines', name=name,
        line=dict(color=color, width=1.5)
    ))

fig.update_layout(
    title='Oil Prices by Region (USD/bbl)',
    xaxis_title='Date', yaxis_title='Price (USD/bbl)',
    template='plotly_white', height=450, hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

In [8]:
# Asia-World spread — when does Asia pay a premium?
asia_spread = prices.dropna(subset=['price_asia'])
asia_spread = asia_spread.copy()
asia_spread['asia_premium'] = asia_spread['price_asia'] - asia_spread['price_world']

print('Asia price premium over world avg:')
print(f'  Overall avg  : ${asia_spread["asia_premium"].mean():.2f}')
print(f'  Max premium  : ${asia_spread["asia_premium"].max():.2f} ({asia_spread["asia_premium"].idxmax().date()})')
print(f'  Max discount : ${asia_spread["asia_premium"].min():.2f} ({asia_spread["asia_premium"].idxmin().date()})')

Asia price premium over world avg:
  Overall avg  : $-0.62
  Max premium  : $5.96 (2012-11-30)
  Max discount : $-9.30 (2004-10-31)


## 5. Rolling Correlation — Price vs Supply and Demand

In [9]:
# 24-month rolling correlation: price vs production and price vs consumption
# Using the master_actuals dataset (starts 1993 where production data begins)
m = master.dropna(subset=['price_world', 'prod_world', 'cons_world']).copy()

window = 24  # 2-year rolling window
m['corr_price_prod'] = m['price_world'].rolling(window).corr(m['prod_world'])
m['corr_price_cons'] = m['price_world'].rolling(window).corr(m['cons_world'])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=m.index, y=m['corr_price_prod'],
    mode='lines', name='Price vs Production',
    line=dict(color='#DC2626', width=1.5)
))
fig.add_trace(go.Scatter(
    x=m.index, y=m['corr_price_cons'],
    mode='lines', name='Price vs Consumption',
    line=dict(color='#059669', width=1.5)
))
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)

fig.update_layout(
    title='24-Month Rolling Correlation: Oil Price vs Production and Consumption',
    xaxis_title='Date',
    yaxis_title='Pearson Correlation',
    yaxis=dict(range=[-1, 1]),
    template='plotly_white', height=420, hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.add_annotation(
    x=0.01, y=0.95, xref='paper', yref='paper',
    text='Positive: price and supply/demand move together<br>Negative: inverse relationship',
    showarrow=False, font=dict(size=11), bgcolor='white', bordercolor='gray'
)
fig.show()

## 6. EIA Price Forecast Through 2027

The EIA Short-Term Energy Outlook (STEO) publishes monthly price forecasts roughly 18-24 months ahead. These are not predictions — they are conditional projections based on current supply, demand, and policy assumptions. They change every month as conditions evolve. Treating them as a directional signal rather than a precise forecast is the right framing.

In [10]:
# Load forecast data
forecast = pd.read_parquet(DATA_DIR + 'master_forecast.parquet')

# The price column in forecast is from the production/consumption join
# For price we use the prices parquet which goes through actuals
# EIA does not provide a standalone price forecast in STEO price sheets
# but we can show where actuals ended and what the supply/demand balance implies

fig = go.Figure()

# Full actual price history
fig.add_trace(go.Scatter(
    x=prices.index, y=prices['price_world'],
    mode='lines', name='World price (actual)',
    line=dict(color='#1D4ED8', width=1.5),
    fill='tozeroy', fillcolor='rgba(37,99,235,0.06)'
))

# 12-month rolling average on actuals
rolling_act = prices['price_world'].rolling(12).mean()
fig.add_trace(go.Scatter(
    x=prices.index, y=rolling_act,
    mode='lines', name='12-month avg (actual)',
    line=dict(color='#F59E0B', width=2, dash='dot')
))

# EIA forecast: supply-demand implied balance shown as shaded context
forecast_balance = forecast['prod_world'] - forecast['cons_world']
forecast_start   = forecast.index.min()
forecast_end     = forecast.index.max()

# Most recent actual price as baseline
last_actual_price = prices['price_world'].dropna().iloc[-1]
last_actual_date  = prices['price_world'].dropna().index[-1]

# Shaded forecast window
fig.add_vrect(
    x0=forecast_start, x1=forecast_end,
    fillcolor='rgba(251,191,36,0.08)',
    layer='below', line_width=0
)
fig.add_vline(x=forecast_start, line_dash='dash',
              line_color='gray', line_width=1.5, opacity=0.6)
fig.add_annotation(
    x=forecast_start, y=120,
    text='EIA forecast period begins',
    showarrow=False, xanchor='left',
    font=dict(size=10, color='gray'), bgcolor='white'
)

# Annotate what balance implies for price direction
avg_fwd_balance = forecast_balance.dropna().mean()
direction_text  = 'EIA projects SURPLUS → price pressure downward' if avg_fwd_balance > 0 else 'EIA projects DEFICIT → price support upward'
fig.add_annotation(
    x=forecast_start + pd.DateOffset(months=6), y=80,
    text=f'{direction_text}<br>Avg balance: {avg_fwd_balance:+.2f} Mb/d',
    showarrow=False, xanchor='left',
    font=dict(size=11), bgcolor='rgba(255,255,200,0.9)',
    bordercolor='gray', borderwidth=1
)

fig.update_layout(
    title='Oil Price History + EIA Supply-Demand Forecast Context (to 2027)',
    xaxis_title='Date', yaxis_title='Price (USD/bbl)',
    xaxis=dict(range=['2010-01-01', str(forecast_end.date())]),
    template='plotly_white', height=500,
    hovermode='x unified',
    legend=dict(orientation='h', y=-0.15)
)
fig.show()

In [11]:
# EIA supply-demand balance forecast by quarter
print('EIA SUPPLY-DEMAND FORECAST (quarterly averages):')
print('Positive = surplus = price pressure. Negative = deficit = price support.')
print()

fcast_q = forecast[['prod_world', 'cons_world']].dropna()
fcast_q['balance'] = fcast_q['prod_world'] - fcast_q['cons_world']

for period, group in fcast_q.resample('QE'):
    avg_balance = group['balance'].mean()
    avg_prod    = group['prod_world'].mean()
    avg_cons    = group['cons_world'].mean()
    direction   = 'SURPLUS' if avg_balance > 0 else 'DEFICIT'
    bar         = '▲' * min(int(abs(avg_balance) * 2), 8) if avg_balance > 0 else '▼' * min(int(abs(avg_balance) * 2), 8)
    quarter = pd.Timestamp(period).quarter
    label = f'{period.year}-Q{quarter}'
    print(f'  {label:12s} Prod={avg_prod:.1f}  Cons={avg_cons:.1f}  Balance={avg_balance:+.2f} Mb/d  {direction} {bar}')

print()
print('NOTE: EIA STEO forecasts are revised monthly. These numbers reflect the')
print('dataset download date. Always verify against the latest STEO release.')

EIA SUPPLY-DEMAND FORECAST (quarterly averages):
Positive = surplus = price pressure. Negative = deficit = price support.

  2026-Q1      Prod=101.8  Cons=103.7  Balance=-1.88 Mb/d  DEFICIT ▼▼▼
  2026-Q2      Prod=105.8  Cons=105.1  Balance=+0.71 Mb/d  SURPLUS ▲
  2026-Q3      Prod=108.3  Cons=106.0  Balance=+2.22 Mb/d  SURPLUS ▲▲▲▲
  2026-Q4      Prod=108.9  Cons=105.6  Balance=+3.30 Mb/d  SURPLUS ▲▲▲▲▲▲
  2027-Q1      Prod=108.8  Cons=105.2  Balance=+3.66 Mb/d  SURPLUS ▲▲▲▲▲▲▲
  2027-Q2      Prod=109.3  Cons=106.7  Balance=+2.64 Mb/d  SURPLUS ▲▲▲▲▲
  2027-Q3      Prod=109.9  Cons=107.4  Balance=+2.44 Mb/d  SURPLUS ▲▲▲▲
  2027-Q4      Prod=110.4  Cons=107.1  Balance=+3.23 Mb/d  SURPLUS ▲▲▲▲▲▲

NOTE: EIA STEO forecasts are revised monthly. These numbers reflect the
dataset download date. Always verify against the latest STEO release.


In [12]:
# Key findings summary — NB02
print('=== NB02 KEY FINDINGS ===')
print()

print('Price extremes (1960-2026):')
print(f'  All-time low  : ${prices["price_world"].min():.2f}/bbl  ({prices["price_world"].idxmin().date()})')
print(f'  All-time high : ${prices["price_world"].max():.2f}/bbl  ({prices["price_world"].idxmax().date()})')
print()

print('3 biggest single-month drops:')
for dt, val in prices['mom_pct'].nsmallest(3).items():
    print(f'  {dt.date()}: {val:+.1f}%')
print()

print('Era with highest avg price:')
best_era = era_stats.loc[era_stats['avg_price'].idxmax()]
print(f'  {best_era["price_era"]} — avg ${best_era["avg_price"]:.2f}/bbl')
print()

print('Most volatile era:')
vol_era = era_stats.loc[era_stats['volatility'].idxmax()]
print(f'  {vol_era["price_era"]} — std dev {vol_era["volatility"]:.1f}%/month')
print()

print('EIA Forecast summary:')
avg_fwd = (forecast['prod_world'] - forecast['cons_world']).dropna().mean()
print(f'  Avg projected balance 2026-2027: {avg_fwd:+.2f} Mb/d')
print(f'  Implied direction: {"SURPLUS — downward price pressure" if avg_fwd > 0 else "DEFICIT — upward price support"}')
print()
print('NB02 complete. Proceed to NB03 — The Shale Revolution.')

=== NB02 KEY FINDINGS ===

Price extremes (1960-2026):
  All-time low  : $1.21/bbl  (1970-01-31)
  All-time high : $132.83/bbl  (2008-07-31)

3 biggest single-month drops:
  2020-03-31: -39.6%
  1986-02-28: -35.5%
  2020-04-30: -34.7%

Era with highest avg price:
  6. High Price Era (2010-2014) — avg $97.67/bbl

Most volatile era:
  2. OPEC Shock Era (1973-1985) — std dev 18.7%/month

EIA Forecast summary:
  Avg projected balance 2026-2027: +2.40 Mb/d
  Implied direction: SURPLUS — downward price pressure

NB02 complete. Proceed to NB03 — The Shale Revolution.
